#### All these were run from google colab with input files present inside drive. 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#!unzip "/content/drive/MyDrive/tamil_poetry_ai.zip" -d /content/

In [ ]:
import sys
sys.path.append("/content/tamil_poetry_ai")

In [ ]:
from src.generator import generate_poem

In [ ]:
#Evaluation

import json
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from collections import defaultdict
import numpy as np
import os

# Import generator and prosody modules
from src.generator import generate_poem
from src.prosody import validate_line

# Set style for plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# output directory
OUTPUT_DIR = ""  # Current directory

# Checking if Google Drive is mounted
if os.path.exists('/content/drive/MyDrive'):
    OUTPUT_DIR = "/content/drive/MyDrive/poetry_evaluation/"
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f" Google Drive detected. Saving to: {OUTPUT_DIR}")
elif os.path.exists('/content'):
    OUTPUT_DIR = "/content/"
    print(f" Google Colab detected. Saving to: {OUTPUT_DIR}")
else:
    OUTPUT_DIR = ""
    print(f" Saving to current directory")


# ==================== CONFIGURATION ====================
TEST_CONFIG = {
    'ஒளவையார்': ['அறம்', 'காதல்', 'இயற்கை', 'வாழ்க்கை'],
    'பாரதிதாசன்': ['அறம்', 'காதல்', 'இயற்கை', 'வாழ்க்கை']
}

POEMS_PER_COMBINATION = 2  # Generate 2 poems for each poet-theme pair
NUM_LINES = 4


# ==================== POET STYLE CHARACTERISTICS ====================
POET_STYLE_PROFILES = {
    'ஒளவையார்': {
        'ideal_word_range': (3, 6),      # Short maxims
        'ideal_line_length': (15, 35),   # Character count
        'weight': 'concise'
    },
    'பாரதிதாசன்': {
        'ideal_word_range': (4, 8),      # Longer poetic lines
        'ideal_line_length': (25, 50),   # Character count
        'weight': 'flowing'
    }
}


# ==================== STYLE VALIDATION ====================
def validate_poet_style(line, poet):
   
    if poet not in POET_STYLE_PROFILES:
        return 0.5  # Neutral if poet unknown

    profile = POET_STYLE_PROFILES[poet]
    score = 0.0

    # Word count check (50% of score)
    word_count = len(line.split())
    ideal_min, ideal_max = profile['ideal_word_range']

    if ideal_min <= word_count <= ideal_max:
        score += 0.5
    elif word_count < ideal_min:
        # Too short - partial credit
        distance = ideal_min - word_count
        score += max(0, 0.5 - (distance * 0.1))
    else:
        # Too long - partial credit
        distance = word_count - ideal_max
        score += max(0, 0.5 - (distance * 0.05))

    # Line length check (50% of score)
    char_count = len(line)
    ideal_min_len, ideal_max_len = profile['ideal_line_length']

    if ideal_min_len <= char_count <= ideal_max_len:
        score += 0.5
    elif char_count < ideal_min_len:
        distance = ideal_min_len - char_count
        score += max(0, 0.5 - (distance * 0.01))
    else:
        distance = char_count - ideal_max_len
        score += max(0, 0.5 - (distance * 0.01))

    return min(score, 1.0)


# ==================== VALIDATION FUNCTIONS ====================
def validate_poem_line(line, poet, line_number=None):

    # Prosody score (0-1)
    prosody_score = validate_line(line, line_number)

    # Style score (0-1)
    style_score = validate_poet_style(line, poet)

    # Combined validation score (50% prosody + 50% style)
    validation_score = (prosody_score + style_score) / 2.0

    validation = {
        'line': line,
        'word_count': len(line.split()),
        'char_count': len(line),
        'prosody_score': prosody_score,
        'style_score': style_score,
        'validation_score': validation_score,  # Single consolidated score
        'passes': validation_score >= 0.6  # Pass threshold
    }

    return validation


def validate_poem(poem, theme, poet):
    """Validate entire poem and return detailed results (NO THEME CHECK)."""
    validations = []

    for i, line in enumerate(poem):
        line_val = validate_poem_line(line, poet, line_number=i+1)
        line_val['line_number'] = i + 1
        validations.append(line_val)

    # Summary - single validation score
    total_lines = len(validations)
    avg_prosody = np.mean([v['prosody_score'] for v in validations])
    avg_style = np.mean([v['style_score'] for v in validations])
    avg_validation = np.mean([v['validation_score'] for v in validations])
    lines_passing = sum(1 for v in validations if v['passes'])

    summary = {
        'total_lines': total_lines,
        'prosody_score': avg_prosody,
        'style_score': avg_style,
        'validation_score': avg_validation,  # Single consolidated score
        'pass_rate': (lines_passing / total_lines * 100) if total_lines > 0 else 0,
        'avg_word_count': np.mean([v['word_count'] for v in validations]),
        'word_count_std': np.std([v['word_count'] for v in validations]),
        'validations': validations
    }

    return summary


# ==================== POEM GENERATION ====================
def generate_test_dataset():

    print("="*80)
    print(" TAMIL POETRY GENERATION - COMPREHENSIVE EVALUATION (v2)")
    print("   Validation: Prosody Score + Poet Style (No Theme Check)")
    print("="*80)
    print(f"\nGenerating {POEMS_PER_COMBINATION} poems for each poet-theme combination...")
    print(f"Total poems to generate: {len(TEST_CONFIG) * 4 * POEMS_PER_COMBINATION}\n")

    all_poems = []
    poem_id = 1

    for poet, themes in TEST_CONFIG.items():
        print(f"\n{'='*80}")
        print(f" POET: {poet}")
        print(f"{'='*80}")

        for theme in themes:
            print(f"\n{'─'*80}")
            print(f" THEME: {theme}")
            print(f"{'─'*80}\n")

            for poem_num in range(1, POEMS_PER_COMBINATION + 1):
                print(f"\n  Poem {poem_num}/{POEMS_PER_COMBINATION}:")
                print(f"  {'─'*76}")

                try:
                    # Generate poem
                    poem = generate_poem(
                        theme=theme,
                        poet=poet,
                        num_lines=NUM_LINES,
                        max_attempts=12
                    )

                    # Validate poem
                    validation = validate_poem(poem, theme, poet)

                    # Print poem
                    print(f"\n   Generated Poem (ID: {poem_id}):")
                    for i, line in enumerate(poem, 1):
                        print(f"     {i}. {line}")

                    # Print validation summary
                    print(f"\n  ✓ Validation Summary:")
                    print(f"     • Prosody Score: {validation['prosody_score']:.3f}")
                    print(f"     • Style Score: {validation['style_score']:.3f}")
                    print(f"     • Validation Score: {validation['validation_score']:.3f} (Combined)")
                    print(f"     • Pass Rate: {validation['pass_rate']:.1f}% (threshold: ≥0.6)")
                    print(f"     • Avg Words/Line: {validation['avg_word_count']:.1f} ± {validation['word_count_std']:.1f}")

                    # Print line-by-line validation
                    print(f"\n   Line-by-Line Validation:")
                    for val in validation['validations']:
                        pass_mark = "✓" if val['passes'] else "✗"

                        print(f"     Line {val['line_number']}: "
                              f"Prosody[{val['prosody_score']:.2f}] "
                              f"Style[{val['style_score']:.2f}] "
                              f"Valid[{val['validation_score']:.2f}] "
                              f"Words[{val['word_count']}] "
                              f"[{pass_mark}]")

                    # Store poem data
                    all_poems.append({
                        'id': poem_id,
                        'poet': poet,
                        'theme': theme,
                        'poem': poem,
                        'validation': validation
                    })

                    poem_id += 1

                except Exception as e:
                    print(f"      Error generating poem: {e}")
                    continue

    return all_poems


# ==================== COMPUTE METRICS BY POET ====================
def compute_poet_metrics(poems_data):

    poet_metrics = {}

    for poet in TEST_CONFIG.keys():
        poet_poems = [p for p in poems_data if p['poet'] == poet]

        if not poet_poems:
            continue

        # Aggregate metrics
        prosody_scores = []
        style_scores = []
        validation_scores = []  # Single consolidated score
        word_counts = []
        lines_passing = 0
        total_lines = 0

        for poem_data in poet_poems:
            val = poem_data['validation']
            total_lines += val['total_lines']
            lines_passing += sum(1 for v in val['validations'] if v['passes'])

            for v in val['validations']:
                prosody_scores.append(v['prosody_score'])
                style_scores.append(v['style_score'])
                validation_scores.append(v['validation_score'])
                word_counts.append(v['word_count'])

        poet_metrics[poet] = {
            'total_poems': len(poet_poems),
            'total_lines': total_lines,
            'prosody_score': np.mean(prosody_scores),
            'style_score': np.mean(style_scores),
            'validation_score': np.mean(validation_scores),  # Single score
            'pass_rate': (lines_passing / total_lines * 100) if total_lines > 0 else 0,
            'avg_words_per_line': np.mean(word_counts),
            'std_words_per_line': np.std(word_counts),
            'min_words': min(word_counts) if word_counts else 0,
            'max_words': max(word_counts) if word_counts else 0
        }

    return poet_metrics


# ==================== COMPUTE CONSOLIDATED METRICS ====================
def compute_consolidated_metrics(poems_data):

    total_poems = len(poems_data)
    prosody_scores = []
    style_scores = []
    validation_scores = []  # Single consolidated score
    word_counts = []
    all_lines = []
    lines_passing = 0
    total_lines = 0

    for poem_data in poems_data:
        val = poem_data['validation']
        total_lines += val['total_lines']
        lines_passing += sum(1 for v in val['validations'] if v['passes'])
        all_lines.extend(poem_data['poem'])

        for v in val['validations']:
            prosody_scores.append(v['prosody_score'])
            style_scores.append(v['style_score'])
            validation_scores.append(v['validation_score'])
            word_counts.append(v['word_count'])

    # Repetition analysis
    unique_lines = len(set(all_lines))
    duplicates = total_lines - unique_lines

    consolidated = {
        'total_poems': total_poems,
        'total_lines': total_lines,
        'prosody_score': np.mean(prosody_scores),
        'style_score': np.mean(style_scores),
        'validation_score': np.mean(validation_scores),  # Single consolidated score
        'pass_rate': (lines_passing / total_lines * 100) if total_lines > 0 else 0,
        'avg_words_per_line': np.mean(word_counts),
        'std_words_per_line': np.std(word_counts),
        'min_words': min(word_counts) if word_counts else 0,
        'max_words': max(word_counts) if word_counts else 0,
        'unique_lines': unique_lines,
        'repetition_rate': (duplicates / total_lines * 100) if total_lines > 0 else 0
    }

    return consolidated


# ==================== PRINT RESULTS ====================
def print_poet_results(poet_metrics):

    print("\n" + "="*80)
    print(" RESULTS BY POET")
    print("="*80)

    for poet, metrics in poet_metrics.items():
        print(f"\n{'─'*80}")
        print(f"  {poet}")
        print(f"{'─'*80}")
        print(f"  Total Poems: {metrics['total_poems']}")
        print(f"  Total Lines: {metrics['total_lines']}")
        print(f"  ")
        print(f"   Prosody Score: {metrics['prosody_score']:.3f} / 1.000")
        print(f"    Style Matching Score: {metrics['style_score']:.3f} / 1.000")
        print(f"   Overall Validation Score: {metrics['validation_score']:.3f} / 1.000")
        print(f"   Pass Rate (≥0.6): {metrics['pass_rate']:.1f}%")
        print(f"  ")
        print(f"  Avg Words/Line: {metrics['avg_words_per_line']:.2f} ± {metrics['std_words_per_line']:.2f}")
        print(f"  Word Count Range: {metrics['min_words']}-{metrics['max_words']} words")


def print_consolidated_results(consolidated):

    print("\n" + "="*80)
    print(" FINAL MODEL VALIDATION RESULTS")
    print("="*80)
    print(f"\n  Dataset:")
    print(f"  {'─'*76}")
    print(f"  Total Poems: {consolidated['total_poems']}")
    print(f"  Total Lines: {consolidated['total_lines']}")
    print(f"  ")
    print(f"  {'='*76}")
    print(f"   OVERALL MODEL VALIDATION SCORE: {consolidated['validation_score']:.3f} / 1.000")
    print(f"  {'='*76}")
    print(f"  ")
    print(f"  Score Breakdown:")
    print(f"      Prosody Compliance: {consolidated['prosody_score']:.3f}")
    print(f"       Poet Style Matching: {consolidated['style_score']:.3f}")
    print(f"  ")
    print(f"  Additional Metrics:")
    print(f"      Pass Rate (≥0.6 threshold): {consolidated['pass_rate']:.1f}%")
    print(f"      Repetition Rate: {consolidated['repetition_rate']:.1f}%")
    print(f"      Unique Lines: {consolidated['unique_lines']}/{consolidated['total_lines']}")
    print(f"  ")
    print(f"  Line Characteristics:")
    print(f"     • Avg Words/Line: {consolidated['avg_words_per_line']:.2f} ± {consolidated['std_words_per_line']:.2f}")
    print(f"     • Range: {consolidated['min_words']}-{consolidated['max_words']} words")
    print(f"  {'─'*76}")


# ==================== CREATE VISUALIZATIONS ====================
def create_visualizations(poems_data, poet_metrics, consolidated):

    print("\n" + "="*80)
    print(" GENERATING VISUALIZATIONS")
    print("="*80)

    poets = list(poet_metrics.keys())

    # Figure 1: Single Validation Score by Poet
    print("\n  Creating: Figure 1 - Validation Score by Poet...")
    fig, ax = plt.subplots(figsize=(12, 6))

    x = np.arange(len(poets))
    width = 0.25

    prosody_scores = [poet_metrics[p]['prosody_score'] for p in poets]
    style_scores = [poet_metrics[p]['style_score'] for p in poets]
    validation_scores = [poet_metrics[p]['validation_score'] for p in poets]

    bars1 = ax.bar(x - width, prosody_scores, width, label='Prosody',
                   color='#3498db', alpha=0.8, edgecolor='black')
    bars2 = ax.bar(x, style_scores, width, label='Style Matching',
                   color='#e74c3c', alpha=0.8, edgecolor='black')
    bars3 = ax.bar(x + width, validation_scores, width, label='Validation Score',
                   color='#27ae60', alpha=0.8, edgecolor='black', linewidth=2)

    ax.set_ylabel('Score (0-1)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Poet', fontsize=12, fontweight='bold')
    ax.set_title('Validation Scores by Poet', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(poets)
    ax.set_ylim(0, 1.0)
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.3)

    # Add value labels
    for bars in [bars1, bars2, bars3]:
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.2f}', ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR + 'fig1_validation_scores.png', dpi=300, bbox_inches='tight')
    print("     ✓ Saved: fig1_validation_scores.png")
    
 """"
    # Figure 2: Pass Rate Comparison
    print("  Creating: Figure 2 - Pass Rate by Poet...")
    fig, ax = plt.subplots(figsize=(10, 6))

    pass_rates = [poet_metrics[p]['pass_rate'] for p in poets]

    bars = ax.bar(poets, pass_rates, color=['#9b59b6', '#f39c12'], alpha=0.8, edgecolor='black')
    ax.axhline(y=consolidated['pass_rate'], color='green', linestyle='--',
               label=f"Overall Average: {consolidated['pass_rate']:.1f}%", linewidth=2)
    ax.axhline(y=60, color='red', linestyle=':', label="Threshold: 60%", linewidth=2)

    ax.set_ylabel('Pass Rate (%)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Poet', fontsize=12, fontweight='bold')
    ax.set_title('Validation Pass Rate by Poet (Score ≥ 0.6)', fontsize=14, fontweight='bold')
    ax.set_ylim(0, 100)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}%', ha='center', va='bottom', fontweight='bold')

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR + 'fig2_pass_rate.png', dpi=300, bbox_inches='tight')
    print("     ✓ Saved: fig2_pass_rate.png")
    """"

    # Figure 3: Word Distribution
    print("  Creating: Figure 3 - Word Count Distribution...")
    fig, ax = plt.subplots(figsize=(12, 6))

    word_counts_by_poet = {}
    for poet in poets:
        poet_poems = [p for p in poems_data if p['poet'] == poet]
        words = []
        for poem_data in poet_poems:
            words.extend([v['word_count'] for v in poem_data['validation']['validations']])
        word_counts_by_poet[poet] = words

    bp = ax.boxplot([word_counts_by_poet[p] for p in poets],
                     labels=poets,
                     patch_artist=True,
                     widths=0.6)

    colors = ['#3498db', '#e74c3c']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    ax.set_ylabel('Words per Line', fontsize=12, fontweight='bold')
    ax.set_xlabel('Poet', fontsize=12, fontweight='bold')
    ax.set_title('Word Count Distribution by Poet', fontsize=14, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR + 'fig3_word_distribution.png', dpi=300, bbox_inches='tight')
    print("     ✓ Saved: fig3_word_distribution.png")

    # Figure 4: Overall Model Validation Score
    print("  Creating: Figure 4 - Overall Model Validation...")

    fig, ax = plt.subplots(figsize=(10, 6))

    # Single validation score as main metric
    validation_pct = consolidated['validation_score'] * 100
    pass_rate = consolidated['pass_rate']
    uniqueness = 100 - consolidated['repetition_rate']

    metrics_names = ['Validation\nScore', 'Pass Rate\n(≥0.6)', 'Uniqueness']
    metrics_values = [validation_pct, pass_rate, uniqueness]

    colors = ['#27ae60', '#3498db', '#9b59b6']
    bars = ax.bar(metrics_names, metrics_values, color=colors, alpha=0.8, edgecolor='black', linewidth=2)

    # Add threshold line at 60%
    ax.axhline(y=60, color='red', linestyle='--', label='Pass Threshold (60%)', linewidth=2, alpha=0.7)

    ax.set_ylabel('Score / Percentage', fontsize=12, fontweight='bold')
    ax.set_title('Overall Model Performance', fontsize=14, fontweight='bold')
    ax.set_ylim(0, 100)
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.3)

    for i, bar in enumerate(bars):
        height = bar.get_height()
        if i == 0:  # Validation score - emphasize
            ax.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.1f}%\n', ha='center', va='bottom',
                    fontweight='bold', fontsize=13, color='green')
        else:
            ax.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.1f}%', ha='center', va='bottom',
                    fontweight='bold', fontsize=11)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR + 'fig4_overall_validation.png', dpi=300, bbox_inches='tight')
    print("     ✓ Saved: fig4_overall_validation.png")

    plt.close('all')
    print("\n   All visualizations created successfully!")


# ==================== SAVE DATA ====================
def save_data(poems_data, poet_metrics, consolidated):

    print("\n" + "="*80)
    print(" SAVING DATA")
    print("="*80)

    # Save poems
    with open(OUTPUT_DIR + 'generated_poems_v2.json', 'w', encoding='utf-8') as f:
        json.dump(poems_data, f, ensure_ascii=False, indent=2)
    print("  ✓ Saved: generated_poems_v2.json")

    # Save metrics
    results = {
        'poet_metrics': poet_metrics,
        'consolidated_metrics': consolidated
    }

    with open(OUTPUT_DIR + 'evaluation_results_v2.json', 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print("  ✓ Saved: evaluation_results_v2.json")


# ==================== MAIN ====================
def main():

    # Step 1: Generate test dataset
    poems_data = generate_test_dataset()

    # Step 2: Compute metrics by poet
    poet_metrics = compute_poet_metrics(poems_data)

    # Step 3: Compute consolidated metrics
    consolidated = compute_consolidated_metrics(poems_data)

    # Step 4: Print results
    print_poet_results(poet_metrics)
    print_consolidated_results(consolidated)

    # Step 5: Create visualizations
    create_visualizations(poems_data, poet_metrics, consolidated)

    # Step 6: Save data
    save_data(poems_data, poet_metrics, consolidated)

    # Final summary
    print("\n" + "="*80)
    print(" EVALUATION COMPLETE")
    print("="*80)
    print(f"\n  Summary:")
    print(f"  • Generated {consolidated['total_poems']} poems ({consolidated['total_lines']} lines)")
    print(f"  ")
    print(f"   OVERALL MODEL VALIDATION SCORE: {consolidated['validation_score']:.3f} / 1.000")
    print(f"  ")
    print(f"  Score Breakdown:")
    print(f"    Prosody: {consolidated['prosody_score']:.3f}")
    print(f"     Style Matching: {consolidated['style_score']:.3f}")
    print(f"  ")
    print(f"  Additional Metrics:")
    print(f"     Pass Rate: {consolidated['pass_rate']:.1f}%")
    print(f"     Repetition: {consolidated['repetition_rate']:.1f}%")
    print(f"  ")
    print(f"  Created 4 visualization figures")
    print(f"  Saved results to JSON files")
    print(f"\n  Validation Methodology:")
    print(f"     - Prosody: Asai-based meter analysis (0-1 score)")
    print(f"     - Style: Poet-specific characteristics matching (0-1 score)")
    print(f"     - Validation Score: (Prosody + Style) / 2")
    print(f"     - Pass threshold: ≥ 0.6 (60%)")
    print(f"     - NO theme checking (only prosody + style)")
    print("="*80 + "\n")


if __name__ == "__main__":
    main()